# TexTag: Baseline CNN (from scratch)

A small CNN trained from scratch. It's the number the pretrained model (step 5) has to beat.

**Run on Google Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.
The first cell clones the repo and the data downloads automatically (~1 min).

Sections:
1. Setup
2. Data
3. Model
4. Train
5. Learning curves
6. Results

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/textag"):
        !git clone -q https://github.com/codecustard/textag.git /content/textag
    %cd /content/textag/notebooks

In [ ]:
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from textag.data import loaders, class_names
from textag.model import SmallCNN, n_params
from textag import engine

torch.manual_seed(42)   # same starting weights every run
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Data

From `textag.data` (step 2): train images get random augmentation, val images don't.
Batch size 32: small enough to fit in memory, big enough for stable updates.

In [ ]:
dl = loaders(batch_size=32, num_workers=2)
classes = class_names()

x, y = next(iter(dl["train"]))
print("one batch of images:", tuple(x.shape), " (batch, channels, height, width)")
print("one batch of labels:", tuple(y.shape))
print({s: len(d.dataset) for s, d in dl.items()})

## 3. Model

A standard CNN design in the VGG style (similar to TinyVGG), kept small because we only have 1,880 training images.

**How it works**

- **Block** = conv 3×3 → batch norm → ReLU → max pool. The conv *looks for patterns*; the max pool *halves the image*.
- **4 blocks stacked.** Each block looks at what the previous one found, so patterns build up from simple to complex.
  The filters are learned during training; we never tell a block what to look for.

| | Image size after | Filters | Typically learns |
|---|---|---|---|
| Input | 224 × 224 | 3 (RGB) | the raw photo |
| Block 1 | 112 × 112 | 32 | edges, color spots |
| Block 2 | 56 × 56 | 64 | small dots, short lines, corners |
| Block 3 | 28 × 28 | 128 | stripes, grid cells, cracks |
| Block 4 | 14 × 14 | 256 | whole patterns: honeycomb, weave |

- **Global average pooling** → 256 numbers: "how much of each pattern is in the image", no matter *where*.
- **2 dense layers** (256 → 128 → 47) turn those into one score per class. **Dropout 0.3** reduces overfitting.

**Why this design**

| Reason | Detail |
|---|---|
| Proven | Follows the VGG block pattern: stacked 3×3 convs + pooling, doubling the filters each block |
| Extends TinyVGG | A bigger version of TinyVGG from [CNN Explainer](https://poloclub.github.io/cnn-explainer/) (2 blocks → 4 blocks), plus batch norm, global average pooling and dropout |
| Fits textures | Conv filters detect small repeating patterns; global average pooling ignores position |
| Fits small data | ~428k weights. Global average pooling cuts the first dense layer from 6.4M weights to 33k; dropout, batch norm and augmentation limit overfitting |
| Purpose | A from-scratch baseline. Numbers (32 filters, 128 hidden, dropout 0.3) are common defaults, not tuned |
| Spec | 4 conv + 2 dense layers (requirement: at least 2) |

**References**

- Simonyan & Zisserman (2014). *Very Deep Convolutional Networks for Large-Scale Image Recognition* (VGG). [arXiv:1409.1556](https://arxiv.org/abs/1409.1556)
- Ioffe & Szegedy (2015). *Batch Normalization*. [arXiv:1502.03167](https://arxiv.org/abs/1502.03167)
- Lin, Chen & Yan (2013). *Network in Network* (global average pooling). [arXiv:1312.4400](https://arxiv.org/abs/1312.4400)
- Srivastava et al. (2014). *Dropout*. [JMLR](https://jmlr.org/papers/v15/srivastava14a.html)
- Zeiler & Fergus (2013). *Visualizing and Understanding Convolutional Networks* (what each layer learns). [arXiv:1311.2901](https://arxiv.org/abs/1311.2901)
- Stanford CS231n notes, *ConvNet Architectures*. [cs231n.github.io/convolutional-networks](https://cs231n.github.io/convolutional-networks/)

In [ ]:
model = SmallCNN(num_classes=len(classes))
print(model)
print(f"trainable parameters: {n_params(model):,}")

## 4. Train

Training loop: `src/textag/engine.py`. `train_step` does one pass over the training data, `val_step` one pass over the validation data,
and `train` runs both every epoch with the LR scheduler and early stopping.

| Choice | Value | Why |
|---|---|---|
| Loss | `CrossEntropyLoss` | Standard for picking 1 class out of many |
| Optimizer | `AdamW`, lr 1e-3, weight decay 1e-4 | Works well without tuning; weight decay reduces overfitting |
| Epochs | up to 40 | Early stopping usually ends it sooner |
| Early stopping | stop after 7 epochs with no val-loss improvement, keep the best epoch | Stops before the model memorizes the training set |
| LR scheduler | `ReduceLROnPlateau`: halve the LR if val loss stalls for 3 epochs | Smaller steps once progress slows |

In [ ]:
from timeit import default_timer as timer

EPOCHS = int(os.environ.get("TEXTAG_EPOCHS", 40))   # env var only used for quick test runs

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3)

start_time = timer()
hist = engine.train(model=model,
                    train_dataloader=dl["train"],
                    val_dataloader=dl["val"],
                    optimizer=optimizer,
                    loss_fn=loss_fn,
                    epochs=EPOCHS,
                    device=device,
                    scheduler=scheduler,
                    patience=7)
end_time = timer()
print(f"Total training time: {end_time - start_time:.1f} seconds")

In [ ]:
# Save the best weights (+ class names, so the app can turn predictions into labels)
MODEL_PATH = ROOT / "models" / "baseline.pt"
MODEL_PATH.parent.mkdir(exist_ok=True)
torch.save({"arch": "small_cnn", "state_dict": model.state_dict(), "classes": classes}, MODEL_PATH)
print("saved", MODEL_PATH.name)

## 5. Learning curves

Train vs val, per epoch. The dashed line marks the best epoch (the weights we keep).
If train keeps improving while val stops, the model is **overfitting** (memorizing the training images).

In [ ]:
h = pd.DataFrame({k: v for k, v in hist.items() if k != "best_epoch"})
h.index = h.index + 1
best = hist["best_epoch"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["loss", "acc"]):
    ax.plot(h.index, h[f"train_{metric}"], label="train")
    ax.plot(h.index, h[f"val_{metric}"], label="val")
    ax.axvline(best, color="gray", linestyle="--", label=f"best epoch ({best})")
    ax.set_xlabel("epoch")
    ax.set_title({"loss": "Loss (lower is better)", "acc": "Accuracy (higher is better)"}[metric])
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Results

In [ ]:
row = h.loc[best]
results = pd.Series({
    "best epoch": best,
    "epochs run": len(h),
    "train accuracy": f"{row.train_acc:.1%}",
    "val accuracy": f"{row.val_acc:.1%}",
    "random guessing": f"{1 / len(classes):.1%}",
    "training time (s)": round(end_time - start_time),
})
results

**Findings:** _filled in after the full run._

**Next:** step 5, a pretrained ResNet18 (transfer learning) trained with the same loop.